# Mini-TP 2 — Actividad entregable (Sesión 2)

**Expón los metadatos de tu modelo por GraphQL y compáralo con REST.**
Individual · entrega esta semana. Completa las celdas marcadas con `# TODO`.

## Consigna
1. Define un **esquema GraphQL** (Strawberry) con un tipo `Model` (`name`, `version`, `metrics`).
2. Una **query** que devuelva las métricas/experimentos de tu modelo (de MLflow local o simuladas).
3. Pruébalo desde **GraphiQL** y desde un **cliente Python**.
4. **Compara** la misma lectura contra tu endpoint **REST** de la Sesión 1 (llamadas y datos) y anota la diferencia.

**Se evalúa:** que corra de punta a punta; que el esquema tipe entrada y salida; que la query pida solo lo necesario; y la reflexión REST vs GraphQL.
**Opcional (+):** que el resolver lea el **linaje** desde Neo4j (ver `graphql_neo4j_lineage.ipynb`).

## 0. Requisitos
```bash
uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests joblib pandas scikit-learn
```

In [1]:
!uv pip install strawberry-graphql fastapi "uvicorn[standard]" requests joblib pandas scikit-learn

Using Python 3.11.16 environment at: C:\Users\Antonella\Documents\Facultad\MaestrÃ­a en IA\MLOpsII\repositorio personal\MLOps2-MIA\.venv
Checked 7 packages in 16ms


## 1. Los metadatos de TU modelo

Reemplaza estos datos por los de tu modelo (o leelos de MLflow).

In [2]:
import json
from pathlib import Path

datos = json.loads(Path("../tp1/modelo/metricas.json").read_text(encoding="utf-8"))

MI_MODELO = {
    "name": datos["nombre"],
    "version": datos["version"],
    "metrics": {
        "auc": datos["metricas"]["roc_auc"],
        "accuracy": datos["metricas"]["accuracy"],
        "f1": datos["metricas"]["f1_score"],
        "precision": datos["metricas"]["precision"],
        "recall": datos["metricas"]["recall"],
    },
}
MI_MODELO

{'name': 'predictor_acv',
 'version': '1.0.0',
 'metrics': {'auc': 0.8642,
  'accuracy': 0.7949,
  'f1': 0.2732,
  'precision': 0.1651,
  'recall': 0.7912}}

## 2. El esquema GraphQL

Completa el tipo `Model` y el resolver que devuelve sus métricas.

In [3]:
import strawberry
from fastapi import FastAPI
from strawberry.fastapi import GraphQLRouter

# --- DEFINICIÓN DE TIPO: Metrics ---
@strawberry.type
class Metrics:
    auc: float
    accuracy: float
    f1: float
    precision: float
    recall: float

# --- DEFINICIÓN DE TIPO: Model ---
@strawberry.type
class Model:
    name: str
    version: str

    # --- DEFINICIÓN DE RELACIÓN: Model -> Metrics ---
    @strawberry.field
    def metrics(self) -> Metrics:
        m = MI_MODELO["metrics"]
        return Metrics(
            auc=m["auc"],
            accuracy=m["accuracy"],
            f1=m["f1"],
            precision=m["precision"],
            recall=m["recall"],
        )

# --- QUERY ---
@strawberry.type
class Query:
    # --- DEFINICIÓN DE QUERY: "model" ---
    @strawberry.field
    def model(self) -> Model:
        return Model(name=MI_MODELO["name"], version=MI_MODELO["version"])

schema = strawberry.Schema(query=Query)
app = FastAPI(title="Mini-TP 2 - metadatos por GraphQL")
app.include_router(GraphQLRouter(schema), prefix="/graphql")
print("Esquema listo")

Esquema listo


## 3. Levantar la API (en segundo plano) y consultar

In [4]:
import threading, time, uvicorn
def _run():
    uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8010, log_level="warning")).run()
threading.Thread(target=_run, daemon=True).start(); time.sleep(2)

import requests

# --- SE CONSULTA AL SERVIDOR LOS METADATOS DEL MODELO (INFO + MÉTRICAS) ---
query = "{ model { name version metrics { auc accuracy f1 precision recall } } }"
r = requests.post("http://127.0.0.1:8010/graphql", json={"query": query})
llamadas_graphql = 1

# --- SE GUARDA EL TAMAÑO DE LA RESPUESTA EN BYTES PARA GRAPHQL. NECESARIO PARA LA COMPARACIÓN CON REST ---
bytes_graphql = len(r.content)
print(r.status_code, r.json())
print(f"GraphQL -> {llamadas_graphql} llamada, {bytes_graphql} bytes")

200 {'data': {'model': {'name': 'predictor_acv', 'version': '1.0.0', 'metrics': {'auc': 0.8642, 'accuracy': 0.7949, 'f1': 0.2732, 'precision': 0.1651, 'recall': 0.7912}}}}
GraphQL -> 1 llamada, 167 bytes


**Probar a mano en GraphiQL:** con la celda de arriba ya corrida (el server queda escuchando en segundo plano), abrí http://127.0.0.1:8010/graphql en el navegador y pegá ahí la misma query:
```graphql
{ model { name version metrics { auc accuracy f1 precision recall } } }
```

In [5]:
# Se observa la flexibilidad de GraphQL frente a REST: con el mismo esquema y el mismo endpoint se puede combinar cualquier 
# conjunto de campos en una sola query, incluso cruzando lo que en REST son dos recursos separados (nombre por un lado, 
# métricas por otro), y hasta pedir una sola métrica en vez de todas.

query_nombre_version = "{ model { name version } }"
query_nombre_una_metrica = "{ model { name metrics { auc } } }"

r1 = requests.post("http://127.0.0.1:8010/graphql", json={"query": query_nombre_version})
r2 = requests.post("http://127.0.0.1:8010/graphql", json={"query": query_nombre_una_metrica})

print("Nombre + versión:", r1.json())
print("Nombre + solo AUC:", r2.json())

Nombre + versión: {'data': {'model': {'name': 'predictor_acv', 'version': '1.0.0'}}}
Nombre + solo AUC: {'data': {'model': {'name': 'predictor_acv', 'metrics': {'auc': 0.8642}}}}


## 4. Comparación con REST  `# TODO`

Sirve el mismo modelo por REST (puedes reusar tu API de la Sesión 1) y cuenta cuántas llamadas y cuántos bytes necesitas para armar la misma vista. Compara con GraphQL.

> Referencia: `rest_vs_graphql.ipynb` hace exactamente esta comparación con código.

In [6]:
# Se levanta la API del TP1 (mini-tps/tp1/api.py) importándola directamente.
# Tiene dos endpoints: /v1/model devuelve los datos del modelo y /v1/model/metrics las métricas.
import sys
sys.path.insert(0, "../tp1")
import api as api_tp1

def _run_rest():
    uvicorn.Server(uvicorn.Config(api_tp1.app, host="127.0.0.1", port=8011, log_level="warning")).run()
threading.Thread(target=_run_rest, daemon=True).start(); time.sleep(2)

r_info = requests.get("http://127.0.0.1:8011/v1/model")
r_metrics = requests.get("http://127.0.0.1:8011/v1/model/metrics")
llamadas_rest = 2

# --- SE GUARDA EL TAMAÑO DE LA RESPUESTA EN BYTES PARA REST ---
bytes_rest = len(r_info.content) + len(r_metrics.content)
print(r_info.status_code, r_info.json())
print(r_metrics.status_code, r_metrics.json())

print(f"\n{'':10}{'llamadas':>10}{'bytes':>10}")
print(f"{'REST':10}{llamadas_rest:>10}{bytes_rest:>10}")
print(f"{'GraphQL':10}{llamadas_graphql:>10}{bytes_graphql:>10}")

200 {'nombre': 'predictor_acv', 'version': '1.0.0', 'umbral': 0.4, 'features': ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status']}
200 {'accuracy': 0.7949, 'precision': 0.1651, 'recall': 0.7912, 'f1_score': 0.2732, 'roc_auc': 0.8642}

            llamadas     bytes
REST               2       292
GraphQL            1       167


Para armar la vista (nombre, versión y métricas), por REST hicieron falta 2 llamadas (una a `/v1/model`, otra a `/v1/model/metrics`) contra 1 sola por GraphQL, que además transfirió menos bytes (167 contra 292). Esa diferencia se debe a que `/v1/model` devuelve también `umbral` y `features`, que no hacían falta para la vista que se pretendía armar pero viajan igual porque REST no permite pedir un subconjunto de campos (over-fetching), mientras que GraphQL solo trajo lo que la query pidió.

## 5. (Opcional +) Linaje desde Neo4j
Si te animas, haz que un resolver devuelva el **linaje** de tu modelo leyendo de Neo4j. Base: `graphql_neo4j_lineage.ipynb`.

---
### Qué entregar
* Este notebook corriendo de punta a punta.
* La query GraphQL y su salida.
* La comparación REST vs GraphQL (números + conclusión).
* En el README de tu repo, una línea sobre qué diferencia notaste.